# Chapter 6.6: Speculative Decoding in SGLang -- Eagle & EAGLE-2

## Learning Objectives

By the end of this notebook, you will:

1. **Understand speculative decoding** fundamentals and why it improves latency
2. **Learn the Eagle architecture** for feature-level draft models
3. **Explore token tree construction** and parallel verification
4. **Understand EAGLE-2 improvements** including dynamic draft length
5. **Trace SGLang's integration** of speculative decoding with the scheduler
6. **Analyze acceptance rates** and their impact on speedup
7. **Walk through source code** for Eagle in SGLang
8. **Configure and evaluate** Eagle speculative decoding

## Prerequisites

- Understanding of autoregressive LLM generation
- Familiarity with the transformer architecture
- Basic probability and statistics

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part6/chapter_6.6_speculative_decoding.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part6/chapter_6.6_speculative_decoding.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. Why Speculative Decoding?

### 1.1 The Autoregressive Bottleneck

Standard LLM generation is **sequential**: each token depends on all previous tokens.

```
Standard autoregressive decoding:
Step 1: [prompt] -> token_1         (1 forward pass)
Step 2: [prompt, token_1] -> token_2 (1 forward pass)
Step 3: [prompt, token_1, token_2] -> token_3 (1 forward pass)
...
Step N: -> token_N (1 forward pass)

Total: N forward passes for N tokens
Each pass: ~5-10ms on GPU (memory-bandwidth bound)
Total latency: N * 5-10ms = very slow for long outputs!
```

### 1.2 The Speculative Decoding Insight

**Key insight**: We can use a small, fast model to "guess" multiple tokens, then **verify all guesses in parallel** with the large model.

```
Speculative decoding:
Step 1: Draft model generates K tokens quickly
  draft: [prompt] -> d1, d2, d3, d4, d5  (fast, ~1ms)

Step 2: Target model verifies ALL K tokens in one pass
  verify: [prompt, d1, d2, d3, d4, d5] -> t1, t2, t3, t4, t5
  (1 forward pass, same cost as 1 token!)

Step 3: Accept matching tokens
  d1=t1? YES (accept)
  d2=t2? YES (accept)
  d3=t3? NO  (reject, use t3 instead)
  -> Generated 3 tokens in ~1 verification pass!

Speedup = tokens_accepted / verification_passes
```

### 1.3 Lossless Guarantee

Speculative decoding can be **lossless** -- the output distribution is exactly the same as the target model alone. This is achieved through a rejection sampling scheme.

In [ ]:
import numpy as np
import time
from typing import List, Tuple

# Basic speculative decoding simulation

class SpeculativeDecodingSimulator:
    """Simulates speculative decoding with acceptance rate."""
    
    def __init__(self, target_time_ms=8.0, draft_time_ms=1.0, acceptance_rate=0.7):
        self.target_time = target_time_ms   # Time for one target model forward pass
        self.draft_time = draft_time_ms     # Time for one draft model forward pass
        self.acceptance_rate = acceptance_rate
    
    def standard_decoding(self, n_tokens):
        """Standard autoregressive: 1 forward pass per token."""
        return n_tokens * self.target_time
    
    def speculative_decoding(self, n_tokens, draft_length=5):
        """Speculative: draft K tokens, verify in parallel."""
        total_time = 0
        tokens_generated = 0
        n_verification_rounds = 0
        
        while tokens_generated < n_tokens:
            # Draft phase: generate K tokens with small model
            total_time += draft_length * self.draft_time
            
            # Verification phase: one target model forward pass
            total_time += self.target_time
            n_verification_rounds += 1
            
            # Accept tokens based on acceptance rate
            accepted = 0
            for i in range(draft_length):
                if np.random.random() < self.acceptance_rate:
                    accepted += 1
                else:
                    accepted += 1  # Accept the corrected token
                    break  # Stop at first rejection
            
            tokens_generated += accepted
        
        return total_time, n_verification_rounds

np.random.seed(42)
sim = SpeculativeDecodingSimulator(target_time_ms=8.0, draft_time_ms=1.0)
n_tokens = 100

standard_time = sim.standard_decoding(n_tokens)

print("=== Speculative Decoding Simulation ===")
print(f"Target model: {sim.target_time}ms/token")
print(f"Draft model:  {sim.draft_time}ms/token")
print(f"Tokens to generate: {n_tokens}\n")

print(f"Standard decoding: {standard_time:.0f} ms ({n_tokens} forward passes)\n")

for accept_rate in [0.5, 0.6, 0.7, 0.8, 0.9]:
    sim.acceptance_rate = accept_rate
    spec_time, n_rounds = sim.speculative_decoding(n_tokens, draft_length=5)
    speedup = standard_time / spec_time
    avg_accepted = n_tokens / n_rounds
    print(f"Spec (accept={accept_rate:.0%}): {spec_time:.0f} ms, "
          f"{n_rounds} rounds, {avg_accepted:.1f} tok/round, speedup={speedup:.2f}x")

## 2. Eagle Architecture

### 2.1 The Feature-Level Draft Model

Traditional speculative decoding uses a separate small language model as the draft model. Eagle takes a different approach: it builds a **feature-level** draft model that operates on the target model's hidden states.

```
Traditional Speculative Decoding:
  Draft model: Independent small LM (e.g., 1B params)
  Pros: Can be any model
  Cons: Separate model weights, lower acceptance rate

Eagle:
  Draft model: Lightweight head on top of target model's features
  Uses: Last hidden state from target model as input
  Pros: Higher acceptance rate, shares target model's understanding
  Cons: Tightly coupled to target model

Architecture:
  Target Model: [Embed] -> [Layer 1] -> ... -> [Layer N] -> [LM Head]
                                                  |
  Eagle Draft:                                    v
                                         [Feature Concat]
                                              |
                                    [Eagle Draft Layers (1-2 layers)]
                                              |
                                    [Draft LM Head]
                                              |
                                    [Draft tokens: d1, d2, ..., dK]
```

### 2.2 Eagle Forward Pass

```python
class EagleDraftModel:
    """
    Eagle uses the target model's hidden states to predict
    the next several tokens.
    """
    def __init__(self, hidden_dim, n_draft_layers=1):
        self.fc = nn.Linear(hidden_dim * 2, hidden_dim)  # Feature fusion
        self.draft_layers = nn.ModuleList([
            TransformerLayer(hidden_dim) for _ in range(n_draft_layers)
        ])
        self.lm_head = nn.Linear(hidden_dim, vocab_size)
    
    def forward(self, target_hidden, token_embeddings):
        """
        Args:
            target_hidden: [batch, seq, hidden] from target model's last layer
            token_embeddings: [batch, seq, hidden] token embeddings
        
        Returns:
            draft_logits: [batch, draft_length, vocab_size]
        """
        # Concatenate hidden state with token embedding
        fused = self.fc(torch.cat([target_hidden, token_embeddings], dim=-1))
        
        # Process through draft layers
        h = fused
        for layer in self.draft_layers:
            h = layer(h)
        
        # Generate draft token logits
        draft_logits = self.lm_head(h)
        return draft_logits
```

In [ ]:
import matplotlib.pyplot as pltimport matplotlib.patches as mpatches# Consistent color palette for all diagramsBLUE = '#4A90D9'GREEN = '#27AE60'ORANGE = '#F39C12'RED = '#E74C3C'PURPLE = '#8E44AD'LIGHT_BLUE = '#85C1E9'LIGHT_GREEN = '#82E0AA'LIGHT_ORANGE = '#F8C471'LIGHT_RED = '#F1948A'LIGHT_PURPLE = '#C39BD3'GRAY = '#95A5A6'DARK_GRAY = '#2C3E50'fig, ax = plt.subplots(1, 1, figsize=(14, 7))ax.set_xlim(0, 14)ax.set_ylim(0, 9)ax.axis('off')fig.patch.set_facecolor('white')ax.text(7, 8.5, 'Eagle Speculative Decoding Architecture', fontsize=14,        fontweight='bold', ha='center', color=DARK_GRAY)# Target modeltarget_box = mpatches.FancyBboxPatch((0.5, 5.5), 5, 2, boxstyle="round,pad=0.15",                                      facecolor=BLUE, alpha=0.85, edgecolor='white', lw=2)ax.add_patch(target_box)ax.text(3, 6.5, 'Target Model (Large)', fontsize=11, fontweight='bold', color='white', ha='center')ax.text(3, 5.9, 'Hidden states from last layer', fontsize=9, color='white', ha='center', style='italic')# Arrow from target to eagleax.annotate('', xy=(7, 6.5), xytext=(5.7, 6.5),            arrowprops=dict(arrowstyle='->', color=DARK_GRAY, lw=2.5))ax.text(6.3, 7, 'hidden\nstates', fontsize=8, color=DARK_GRAY, ha='center')# Eagle headeagle_box = mpatches.FancyBboxPatch((7, 5.5), 3.5, 2, boxstyle="round,pad=0.15",                                     facecolor=GREEN, alpha=0.85, edgecolor='white', lw=2)ax.add_patch(eagle_box)ax.text(8.75, 6.5, 'Eagle Draft Head', fontsize=11, fontweight='bold', color='white', ha='center')ax.text(8.75, 5.9, 'Small MLP (1-2 layers)', fontsize=9, color='white', ha='center', style='italic')# Arrow from eagle to draft tokensax.annotate('', xy=(8.75, 5), xytext=(8.75, 5.5),            arrowprops=dict(arrowstyle='->', color=DARK_GRAY, lw=2.5))# Draft tokensdraft_box = mpatches.FancyBboxPatch((6.5, 3.5), 4.5, 1.2, boxstyle="round,pad=0.1",                                     facecolor=ORANGE, alpha=0.85, edgecolor='white', lw=2)ax.add_patch(draft_box)ax.text(8.75, 4.1, 'Draft Tokens: d1, d2, d3, d4, d5', fontsize=10,        fontweight='bold', color='white', ha='center')# Arrow back to target for verificationax.annotate('', xy=(3, 5.5), xytext=(7, 3.8),            arrowprops=dict(arrowstyle='->', color=RED, lw=2.5,                           connectionstyle='arc3,rad=0.3'))ax.text(3.5, 4.2, 'Verify with\ntarget model', fontsize=9, color=RED, ha='center', fontweight='bold')# Verification resultverify_box = mpatches.FancyBboxPatch((0.5, 1.5), 5, 1.5, boxstyle="round,pad=0.15",                                      facecolor=RED, alpha=0.85, edgecolor='white', lw=2)ax.add_patch(verify_box)ax.text(3, 2.5, 'Verify All at Once', fontsize=11, fontweight='bold', color='white', ha='center')ax.text(3, 1.9, 'd1=OK, d2=OK, d3=REJECT -> use t3', fontsize=9, color='white', ha='center')# Resultax.text(8.75, 2, '3 tokens in 1 pass!\n(vs 3 separate passes)', ha='center', fontsize=11,        color=GREEN, fontweight='bold',        bbox=dict(boxstyle='round,pad=0.3', facecolor=LIGHT_GREEN, alpha=0.4))plt.tight_layout()plt.savefig("eagle_architecture.png", dpi=150, bbox_inches='tight', facecolor='white')plt.show()

In [ ]:
import torch
import torch.nn as nn

class SimpleEagleDraft(nn.Module):
    """Simplified Eagle draft model for demonstration."""
    
    def __init__(self, hidden_dim, vocab_size, n_draft_layers=1):
        super().__init__()
        # Feature fusion: combine target hidden + token embedding
        self.fc_in = nn.Linear(hidden_dim * 2, hidden_dim)
        
        # Lightweight draft transformer layers
        self.draft_layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=hidden_dim, nhead=8, dim_feedforward=hidden_dim*2,
                batch_first=True, dropout=0.0
            ) for _ in range(n_draft_layers)
        ])
        
        # Draft language model head
        self.lm_head = nn.Linear(hidden_dim, vocab_size, bias=False)
    
    def forward(self, target_hidden, token_emb):
        """
        Generate draft predictions from target model's features.
        
        target_hidden: [batch, 1, hidden_dim] - target model's last layer output
        token_emb: [batch, 1, hidden_dim] - embedding of last generated token
        """
        # Fuse features
        fused = self.fc_in(torch.cat([target_hidden, token_emb], dim=-1))
        fused = torch.nn.functional.silu(fused)
        
        # Apply draft layers
        h = fused
        for layer in self.draft_layers:
            h = layer(h)
        
        # Get logits
        logits = self.lm_head(h)
        return logits

# Create a demo
hidden_dim = 256
vocab_size = 1000
device = 'cuda' if torch.cuda.is_available() else 'cpu'

eagle_draft = SimpleEagleDraft(hidden_dim, vocab_size, n_draft_layers=1).to(device)

# Simulated target model output
target_hidden = torch.randn(1, 1, hidden_dim, device=device)
token_emb = torch.randn(1, 1, hidden_dim, device=device)

with torch.no_grad():
    draft_logits = eagle_draft(target_hidden, token_emb)

draft_params = sum(p.numel() for p in eagle_draft.parameters())
print(f"Eagle Draft Model:")
print(f"  Parameters: {draft_params:,} ({draft_params/1e6:.1f}M)")
print(f"  Input: target_hidden {target_hidden.shape} + token_emb {token_emb.shape}")
print(f"  Output: draft_logits {draft_logits.shape}")
print(f"  Top-5 draft tokens: {draft_logits.argmax(dim=-1).item()}")

## 3. Token Tree Construction and Verification

### 3.1 From Linear to Tree Speculation

Instead of generating a single chain of draft tokens, Eagle generates a **tree** of possibilities:

```
Linear speculation:
  d1 -> d2 -> d3 -> d4 -> d5
  If d2 is wrong, d3-d5 are all wasted.

Tree speculation:
                d1
              /    \
           d2a      d2b       (top-2 at position 2)
           / \       |
         d3a d3b    d3c       (top-2 and top-1)
          |   |      |
         d4a d4b    d4c       (top-1 at position 4)
  
  Even if d2a is wrong, d2b might be right!
  More chances for acceptance = higher throughput.
```

### 3.2 Tree Verification

The tree is verified in a single forward pass using **tree attention**:

```
Tree attention mask:
       d1  d2a d2b d3a d3b d3c d4a d4b d4c
d1   [  1   0   0   0   0   0   0   0   0 ]
d2a  [  1   1   0   0   0   0   0   0   0 ]
d2b  [  1   0   1   0   0   0   0   0   0 ]
d3a  [  1   1   0   1   0   0   0   0   0 ]
d3b  [  1   1   0   0   1   0   0   0   0 ]
d3c  [  1   0   1   0   0   1   0   0   0 ]
d4a  [  1   1   0   1   0   0   1   0   0 ]
d4b  [  1   1   0   0   1   0   0   1   0 ]
d4c  [  1   0   1   0   0   1   0   0   1 ]

Each row attends only to its ancestors in the tree.
This allows verifying ALL paths in ONE forward pass!
```

In [ ]:
# Implement token tree for speculative decoding

class TokenTree:
    """Token tree for tree-based speculative decoding."""
    
    def __init__(self):
        self.nodes = []  # List of (token_id, parent_idx, depth)
        self.children = {}  # node_idx -> list of child indices
    
    def add_root(self, token_id):
        idx = len(self.nodes)
        self.nodes.append({"token": token_id, "parent": -1, "depth": 0})
        self.children[idx] = []
        return idx
    
    def add_child(self, parent_idx, token_id):
        idx = len(self.nodes)
        depth = self.nodes[parent_idx]["depth"] + 1
        self.nodes.append({"token": token_id, "parent": parent_idx, "depth": depth})
        self.children[idx] = []
        if parent_idx not in self.children:
            self.children[parent_idx] = []
        self.children[parent_idx].append(idx)
        return idx
    
    def get_attention_mask(self):
        """Generate tree attention mask."""
        n = len(self.nodes)
        mask = np.zeros((n, n), dtype=np.bool_)
        
        for i in range(n):
            # Each node attends to itself and all ancestors
            idx = i
            while idx >= 0:
                mask[i, idx] = True
                idx = self.nodes[idx]["parent"]
        
        return mask
    
    def get_paths(self):
        """Get all root-to-leaf paths."""
        paths = []
        leaves = [i for i in range(len(self.nodes)) if i not in self.children or not self.children[i]]
        
        for leaf in leaves:
            path = []
            idx = leaf
            while idx >= 0:
                path.append(self.nodes[idx]["token"])
                idx = self.nodes[idx]["parent"]
            paths.append(list(reversed(path)))
        
        return paths
    
    def print_tree(self, idx=0, indent=0):
        node = self.nodes[idx]
        print(" " * indent + f"Token: {node['token']} (depth={node['depth']})")
        for child_idx in self.children.get(idx, []):
            self.print_tree(child_idx, indent + 2)

# Build example tree
tree = TokenTree()
r = tree.add_root(token_id=42)  # Root: first draft token

# Level 1: top-2 candidates
c1 = tree.add_child(r, token_id=15)   # Most likely
c2 = tree.add_child(r, token_id=23)   # Second most likely

# Level 2: top-2 for first branch, top-1 for second
c3 = tree.add_child(c1, token_id=7)   
c4 = tree.add_child(c1, token_id=31)  
c5 = tree.add_child(c2, token_id=12)  

# Level 3: top-1 for each
tree.add_child(c3, token_id=55)
tree.add_child(c4, token_id=88)
tree.add_child(c5, token_id=44)

print("=== Token Tree ===")
tree.print_tree()

print(f"\n=== Tree Attention Mask ===")
mask = tree.get_attention_mask()
print(f"Size: {mask.shape}")
print(mask.astype(int))

print(f"\n=== Root-to-Leaf Paths ===")
for path in tree.get_paths():
    print(f"  {path}")

In [ ]:
# Simulate tree verification

def verify_tree(tree, target_model_predictions):
    """
    Verify a token tree against target model predictions.
    
    Args:
        tree: TokenTree with draft tokens
        target_model_predictions: dict mapping node_idx -> target model's top prediction
    
    Returns:
        accepted_path: longest accepted path from root
        accepted_length: number of accepted tokens
    """
    # Start from root and greedily accept matching tokens
    accepted = []
    current_idx = 0  # Start at root
    
    while True:
        node = tree.nodes[current_idx]
        target_token = target_model_predictions.get(current_idx)
        
        if target_token is None:
            break
        
        if node["token"] == target_token:
            # Accept this token
            accepted.append(node["token"])
            
            # Try children
            children = tree.children.get(current_idx, [])
            found_match = False
            for child_idx in children:
                child_target = target_model_predictions.get(child_idx)
                if child_target is not None and tree.nodes[child_idx]["token"] == child_target:
                    current_idx = child_idx
                    found_match = True
                    break
            
            if not found_match:
                # No matching child, but this node was accepted
                # Add the target's prediction as the next token
                if children:
                    # The target model gives us the correct next token
                    accepted.append(target_model_predictions[children[0]])
                break
        else:
            # Reject: use target's token instead
            accepted.append(target_token)
            break
    
    return accepted

# Simulate verification scenarios
print("=== Tree Verification Scenarios ===\n")

# Scenario 1: High acceptance (draft model is accurate)
high_accept = {
    0: 42,  # Root matches
    1: 15,  # Child 1 matches (token 15)
    2: 23,  # Child 2 also matches but we go with child 1
    3: 7,   # Grandchild matches
    4: 31,  
    5: 12,
    6: 55,  # Great-grandchild matches
    7: 88,
    8: 44,
}
result1 = verify_tree(tree, high_accept)
print(f"High acceptance: {result1} ({len(result1)} tokens in 1 verification pass)")

# Scenario 2: Rejection at level 2
mid_accept = {
    0: 42,  # Root matches
    1: 15,  # Child 1 matches
    2: 23,
    3: 99,  # Grandchild MISMATCH (draft said 7, target says 99)
    4: 31,
    5: 12,
    6: 55,
    7: 88,
    8: 44,
}
result2 = verify_tree(tree, mid_accept)
print(f"Mid acceptance:  {result2} ({len(result2)} tokens in 1 verification pass)")

# Scenario 3: Immediate rejection
low_accept = {
    0: 99,  # Root MISMATCH (draft said 42, target says 99)
    1: 15,
    2: 23,
}
result3 = verify_tree(tree, low_accept)
print(f"Low acceptance:  {result3} ({len(result3)} tokens in 1 verification pass)")

## 4. EAGLE-2 Improvements

### 4.1 Dynamic Draft Length

EAGLE-1 uses a fixed draft length. EAGLE-2 adapts the draft length based on confidence:

```
EAGLE-1: Always draft K=5 tokens
  High confidence: drafts 5 tokens, all accepted  -> efficient
  Low confidence:  drafts 5 tokens, 1 accepted    -> wasted effort

EAGLE-2: Dynamic draft length based on confidence
  High confidence: drafts 8 tokens (extend for more savings)
  Low confidence:  drafts 2 tokens (don't waste compute)
```

### 4.2 Confidence Estimation

```python
def estimate_confidence(draft_logits):
    """
    Estimate how likely the draft tokens are to be accepted.
    
    High entropy = low confidence = short draft
    Low entropy = high confidence = long draft
    """
    probs = softmax(draft_logits, dim=-1)
    entropy = -(probs * probs.log()).sum(dim=-1)
    
    # Map entropy to draft length
    if entropy < 0.5:
        return 8  # Very confident: long draft
    elif entropy < 1.0:
        return 5  # Moderate confidence
    elif entropy < 2.0:
        return 3  # Low confidence
    else:
        return 1  # Very uncertain: minimal draft
```

### 4.3 Context-Aware Tree Structure

EAGLE-2 also adapts the tree shape based on context:

```
High confidence context (e.g., continuing a pattern):
  Deep, narrow tree (maximize path length)
  d1 -> d2 -> d3 -> d4 -> d5 -> d6 -> d7 -> d8

Low confidence context (e.g., creative writing):
  Shallow, wide tree (maximize chances of one correct path)
            d1
         /  |  \
       d2a d2b d2c
       |   |   |
      d3a d3b d3c
```

In [ ]:
# Simulate EAGLE-2 dynamic draft length

class EAGLE2Simulator:
    """Simulates EAGLE-2 with dynamic draft length."""
    
    def __init__(self, target_time_ms=8.0, draft_time_ms=0.5):
        self.target_time = target_time_ms
        self.draft_time = draft_time_ms
    
    def estimate_acceptance_rate(self, context_type):
        """Simulate varying acceptance rates by context."""
        rates = {
            "factual": 0.85,       # High: predictable content
            "code": 0.80,          # High: structured
            "continuation": 0.75,   # Medium: following pattern
            "creative": 0.50,       # Low: unpredictable
            "reasoning": 0.60,      # Medium-low: complex logic
        }
        return rates.get(context_type, 0.7)
    
    def dynamic_draft_length(self, confidence):
        """EAGLE-2: adapt draft length based on confidence."""
        if confidence > 0.8:
            return 8
        elif confidence > 0.7:
            return 6
        elif confidence > 0.6:
            return 4
        elif confidence > 0.5:
            return 3
        else:
            return 2
    
    def simulate_generation(self, n_tokens, context_type, use_dynamic=True):
        """Generate n_tokens with Eagle-2."""
        acceptance_rate = self.estimate_acceptance_rate(context_type)
        total_time = 0
        tokens_generated = 0
        total_drafted = 0
        n_rounds = 0
        
        while tokens_generated < n_tokens:
            if use_dynamic:
                draft_len = self.dynamic_draft_length(acceptance_rate)
            else:
                draft_len = 5  # Fixed length (Eagle-1)
            
            # Draft phase
            total_time += draft_len * self.draft_time
            total_drafted += draft_len
            
            # Verification phase
            total_time += self.target_time
            n_rounds += 1
            
            # Acceptance simulation
            accepted = 0
            for i in range(draft_len):
                if np.random.random() < acceptance_rate:
                    accepted += 1
                else:
                    accepted += 1  # Corrected token
                    break
            
            tokens_generated += accepted
        
        return {
            "total_time": total_time,
            "tokens_generated": tokens_generated,
            "total_drafted": total_drafted,
            "n_rounds": n_rounds,
            "avg_accepted": tokens_generated / n_rounds,
            "tokens_per_ms": tokens_generated / total_time,
        }

np.random.seed(42)
sim2 = EAGLE2Simulator(target_time_ms=8.0, draft_time_ms=0.5)
n_tokens = 200

contexts = ["factual", "code", "continuation", "reasoning", "creative"]

print("=== EAGLE-1 (Fixed) vs EAGLE-2 (Dynamic) ===")
print(f"Target: {sim2.target_time}ms, Draft: {sim2.draft_time}ms, Tokens: {n_tokens}\n")

baseline_time = n_tokens * sim2.target_time

print(f"{'Context':<15s} | {'Accept Rate':>11s} | {'E1 Time':>10s} | {'E2 Time':>10s} | "
      f"{'E1 Speedup':>10s} | {'E2 Speedup':>10s} | {'E2 vs E1':>10s}")
print("-" * 90)

for ctx in contexts:
    rate = sim2.estimate_acceptance_rate(ctx)
    e1 = sim2.simulate_generation(n_tokens, ctx, use_dynamic=False)
    e2 = sim2.simulate_generation(n_tokens, ctx, use_dynamic=True)
    
    e1_speedup = baseline_time / e1["total_time"]
    e2_speedup = baseline_time / e2["total_time"]
    e2_vs_e1 = e1["total_time"] / e2["total_time"]
    
    print(f"{ctx:<15s} | {rate:>10.0%} | {e1['total_time']:>9.0f} | {e2['total_time']:>9.0f} | "
          f"{e1_speedup:>9.2f}x | {e2_speedup:>9.2f}x | {e2_vs_e1:>9.2f}x")

## 5. Integration with SGLang Scheduler

### 5.1 Architecture

```
SGLang with Speculative Decoding:

    Scheduler
       |
       v
    +---------------------------+
    | Model Runner              |
    |  +---------------------+  |
    |  | Target Model (Large)|  |
    |  +---------------------+  |
    |  | Eagle Draft Model   |  |
    |  +---------------------+  |
    +---------------------------+
       |
       v
    Decode Loop:
    1. Eagle draft: generate K token candidates
    2. Build token tree with top-K at each level
    3. Target model: verify tree in one pass
    4. Accept matching tokens
    5. Update KV-cache with accepted tokens
    6. Repeat from step 1
```

### 5.2 Source Code: Eagle Integration

```python
# From sglang/srt/speculative/eagle_worker.py

class EagleWorker:
    """
    Manages Eagle speculative decoding.
    Works alongside the main model runner.
    """
    
    def __init__(self, model_runner, eagle_model):
        self.model_runner = model_runner  # Target model
        self.eagle_model = eagle_model    # Eagle draft model
        self.num_speculative_tokens = 5   # Default draft length
    
    def generate_draft(self, input_ids, hidden_states):
        """
        Generate draft tokens using Eagle model.
        
        Uses the target model's hidden states as input,
        producing a tree of candidate tokens.
        """
        draft_tokens = []
        draft_tree = TokenTree()
        
        # Use target model's last hidden state
        h = hidden_states[:, -1:, :]  # Last position
        
        for depth in range(self.num_speculative_tokens):
            # Get token embeddings
            if depth == 0:
                token_emb = self.embed(input_ids[:, -1:])
            else:
                token_emb = self.embed(draft_tokens[-1])
            
            # Eagle forward pass
            logits = self.eagle_model(h, token_emb)
            
            # Select top candidates
            top_k = min(3, logits.shape[-1])  # Top-3 candidates
            top_tokens = logits.topk(top_k, dim=-1)
            
            draft_tokens.append(top_tokens.indices)
            # Update hidden state for next step
            h = self.eagle_model.get_hidden(h, token_emb)
        
        return draft_tree, draft_tokens
    
    def verify_and_accept(self, draft_tree, target_logits):
        """
        Verify draft tokens against target model output.
        Returns accepted tokens and their count.
        """
        accepted_tokens = []
        
        for depth in range(len(draft_tree.nodes)):
            draft_token = draft_tree.nodes[depth]["token"]
            target_token = target_logits[depth].argmax().item()
            
            if draft_token == target_token:
                accepted_tokens.append(target_token)
            else:
                accepted_tokens.append(target_token)  # Use target's token
                break  # Stop at first rejection
        
        return accepted_tokens
```

In [ ]:
# Analyze acceptance rate impact on speedup

def theoretical_speedup(acceptance_rate, draft_length, draft_cost_ratio=0.1):
    """
    Calculate theoretical speedup from speculative decoding.
    
    Args:
        acceptance_rate: probability of each draft token being accepted
        draft_length: number of draft tokens per round
        draft_cost_ratio: cost of drafting / cost of verification
    
    Returns:
        speedup over standard autoregressive decoding
    """
    # Expected accepted tokens per round (geometric distribution)
    expected_accepted = 0
    for i in range(draft_length):
        # Probability of accepting exactly i tokens then rejecting
        if i < draft_length - 1:
            prob = (acceptance_rate ** i) * (1 - acceptance_rate)
        else:
            prob = acceptance_rate ** i  # All accepted
        expected_accepted += (i + 1) * prob  # +1 for the corrected token
    
    # Cost per round: drafting + verification
    cost_per_round = draft_length * draft_cost_ratio + 1.0
    
    # Tokens per unit cost
    tokens_per_cost = expected_accepted / cost_per_round
    
    # Standard decoding: 1 token per unit cost
    return tokens_per_cost

print("=== Theoretical Speedup Analysis ===\n")

# Table: acceptance rate vs draft length vs speedup
rates = [0.5, 0.6, 0.7, 0.8, 0.9, 0.95]
lengths = [2, 3, 5, 8, 12]

print(f"{'Draft Length':>12s}", end="")
for r in rates:
    print(f" | {r:.0%}:>8s", end="")
print()
print("-" * 70)

for length in lengths:
    print(f"{length:>12d}", end="")
    for rate in rates:
        speedup = theoretical_speedup(rate, length, draft_cost_ratio=0.1)
        print(f" | {speedup:>7.2f}x", end="")
    print()

print("\nKey insights:")
print("  1. Higher acceptance rate -> more benefit from longer drafts")
print("  2. At low acceptance rates, shorter drafts waste less compute")
print("  3. Sweet spot depends on the acceptance_rate * draft_cost trade-off")

In [ ]:
# Visualize speedup heatmap
try:
    import matplotlib.pyplot as plt
    HAS_PLT = True
except ImportError:
    HAS_PLT = False

if HAS_PLT:
    rates_fine = np.linspace(0.3, 0.99, 50)
    lengths_fine = range(1, 16)
    
    speedup_matrix = np.zeros((len(lengths_fine), len(rates_fine)))
    for i, length in enumerate(lengths_fine):
        for j, rate in enumerate(rates_fine):
            speedup_matrix[i, j] = theoretical_speedup(rate, length, draft_cost_ratio=0.1)
    
    fig, ax = plt.subplots(figsize=(12, 6))
    im = ax.imshow(speedup_matrix, aspect='auto', origin='lower',
                   extent=[rates_fine[0]*100, rates_fine[-1]*100, 
                           min(lengths_fine), max(lengths_fine)],
                   cmap='RdYlGn', vmin=0.5, vmax=5.0)
    
    # Add contour lines
    cs = ax.contour(rates_fine * 100, list(lengths_fine), speedup_matrix,
                     levels=[1.0, 1.5, 2.0, 2.5, 3.0, 4.0], colors='black', linewidths=0.5)
    ax.clabel(cs, inline=True, fontsize=8, fmt='%.1fx')
    
    ax.set_xlabel('Acceptance Rate (%)')
    ax.set_ylabel('Draft Length')
    ax.set_title('Speculative Decoding Speedup (draft_cost_ratio=0.1)')
    plt.colorbar(im, label='Speedup')
    plt.tight_layout()
    plt.show()

## 6. SGLang Configuration

### 6.1 Launching with Eagle

```bash
# Launch SGLang with Eagle speculative decoding
python -m sglang.launch_server \
    --model meta-llama/Llama-3-8B-Instruct \
    --speculative-algorithm eagle \
    --speculative-draft-model-path /path/to/eagle-draft-model \
    --speculative-num-draft-tokens 5 \
    --speculative-eagle-topk 4
```

### 6.2 Configuration Parameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| `--speculative-algorithm` | None | `eagle` or `eagle2` |
| `--speculative-draft-model-path` | None | Path to Eagle draft model |
| `--speculative-num-draft-tokens` | 5 | Number of draft tokens per round |
| `--speculative-eagle-topk` | 4 | Top-K candidates at each tree level |
| `--speculative-num-steps` | 5 | Max tree depth |

## 7. Summary

### Key Takeaways

1. **Speculative decoding** reduces latency by drafting multiple tokens with a fast model and verifying them in parallel.

2. **Eagle** uses the target model's hidden states as input to a lightweight draft model, achieving higher acceptance rates than independent draft models.

3. **Token trees** enable exploring multiple draft paths simultaneously, increasing the probability of finding an accepted sequence.

4. **EAGLE-2** dynamically adjusts draft length based on confidence, optimizing the compute trade-off.

5. **Acceptance rate** is the critical metric. Higher rates benefit from longer drafts; lower rates should use shorter drafts.

6. **Typical speedups** are 1.5-3x for greedy decoding, depending on the task and acceptance rate.

7. Speculative decoding is **most beneficial** for latency-sensitive scenarios with a single request (batch size 1).

## Exercises

### Exercise 1: Analyze Speculation Efficiency

For different text types (code, prose, math), measure:
1. Token prediction entropy at each position
2. Simulated acceptance rate
3. Optimal draft length for each text type

### Exercise 2: Implement Rejection Sampling

Implement the full rejection sampling algorithm that ensures lossless speculative decoding:
- Draft model distribution: q(x)
- Target model distribution: p(x)
- Accept with probability min(1, p(x)/q(x))

### Exercise 3: Tree Shape Optimization

Given a fixed budget of N total tree nodes, find the optimal tree shape (depth vs width) that maximizes expected accepted tokens for different acceptance rates.

### Exercise 4: Configure Eagle for Your Use Case

If you have GPU access, serve a model with Eagle and:
1. Measure speedup on different prompt types
2. Find the optimal `speculative-num-draft-tokens`
3. Compare latency at batch_size=1 vs batch_size=32

In [ ]:
# Exercise 2 Starter: Rejection Sampling for Lossless Speculative Decoding

def rejection_sampling_verify(draft_probs, target_probs, draft_token):
    """
    TODO: Implement rejection sampling verification.
    
    The algorithm ensures the output distribution matches the target exactly:
    
    1. Accept draft token with probability min(1, p(x)/q(x))
       where p = target distribution, q = draft distribution
    2. If rejected, sample from the residual distribution:
       r(x) = max(0, p(x) - q(x)) / sum(max(0, p(x) - q(x)))
    
    Args:
        draft_probs: [vocab_size] draft model probabilities
        target_probs: [vocab_size] target model probabilities  
        draft_token: int, the token proposed by the draft model
    
    Returns:
        accepted: bool
        final_token: int, the accepted or resampled token
    """
    # YOUR CODE HERE
    # Step 1: Compute acceptance probability
    p = target_probs[draft_token]
    q = draft_probs[draft_token]
    accept_prob = min(1.0, p / (q + 1e-10))
    
    # Step 2: Accept or reject
    if np.random.random() < accept_prob:
        return True, draft_token
    else:
        # Sample from residual distribution
        residual = np.maximum(0, target_probs - draft_probs)
        residual_sum = residual.sum()
        if residual_sum > 0:
            residual /= residual_sum
            resampled = np.random.choice(len(target_probs), p=residual)
        else:
            resampled = np.random.choice(len(target_probs), p=target_probs)
        return False, resampled

# Test
vocab_size = 10
draft_probs = np.array([0.3, 0.2, 0.15, 0.1, 0.08, 0.07, 0.05, 0.03, 0.01, 0.01])
target_probs = np.array([0.25, 0.22, 0.18, 0.12, 0.08, 0.06, 0.04, 0.03, 0.01, 0.01])

# Run many trials to verify the output distribution matches target
n_trials = 10000
accepted_count = 0
output_counts = np.zeros(vocab_size)

for _ in range(n_trials):
    draft_token = np.random.choice(vocab_size, p=draft_probs)
    accepted, final_token = rejection_sampling_verify(draft_probs, target_probs, draft_token)
    output_counts[final_token] += 1
    if accepted:
        accepted_count += 1

output_distribution = output_counts / n_trials
print("Rejection Sampling Verification:")
print(f"  Acceptance rate: {accepted_count/n_trials:.1%}")
print(f"  Target distribution:  {target_probs}")
print(f"  Output distribution:  {np.round(output_distribution, 3)}")
print(f"  Max absolute error:   {np.abs(target_probs - output_distribution).max():.4f}")
print(f"  Distribution match:   {'PASS' if np.abs(target_probs - output_distribution).max() < 0.03 else 'FAIL'}")

In [ ]:
# Exercise 3 Starter: Tree Shape Optimization

def evaluate_tree_shape(total_nodes, branching_factors, acceptance_rate):
    """
    TODO: Evaluate a tree shape given a budget of total nodes.
    
    Args:
        total_nodes: Maximum nodes in the tree
        branching_factors: List of branching factor at each depth
                          e.g., [3, 2, 1] means 3 branches at depth 1,
                          2 at depth 2, 1 at depth 3
        acceptance_rate: Probability of each draft token being correct
    
    Returns:
        expected_accepted: Expected number of accepted tokens
    """
    # YOUR CODE HERE
    # Compute expected accepted tokens for this tree shape
    # Hint: At each depth, the probability of at least one branch matching is:
    #   1 - (1 - acceptance_rate)^branching_factor
    pass

print("Exercise 3: Implement evaluate_tree_shape.")
print("Goal: Find the best tree shape for a budget of 15 nodes.")
print("\nExample tree shapes:")
print("  [1,1,1,...,1] = deep narrow (15 levels deep, 1 branch each)")
print("  [3,3,1] = wide then narrow (3*3 + 3 + 1 = 13 nodes at 3 levels)")
print("  [5,3] = very wide shallow (5*3 + 5 + 1 = 21, too many -- adjust)")